# Dashboard Analysis: US Youth Soccer Cost-Benefit Model
**Real Data Cost-Benefit Analysis & ROI Modeling**

Source: MLSPA 2026, NCAA 2025-26, MLS NEXT enrollment, research statistics (202 data points)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Load all datasets
costs = pd.read_csv('../data/processed/costs.csv')
programs = pd.read_csv('../data/processed/programs.csv')
scholarships = pd.read_csv('../data/processed/scholarship_outcomes.csv')
pro_outcomes = pd.read_csv('../data/processed/pro_outcomes.csv')
mls_salaries = pd.read_csv('../data/processed/mls_player_salaries_2026.csv')
research = pd.read_csv('../data/processed/research_statistics.csv')

print(f'Loaded {len(costs)} cost records')
print(f'Loaded {len(programs)} program records')
print(f'Loaded {len(scholarships)} scholarship records')
print(f'Loaded {len(mls_salaries)} MLS player records')
print(f'Loaded {len(research)} research statistics')

## 1. THE FUNNEL: Probability of Success by Pathway

In [ ]:
# Probability funnel based on research statistics
funnel_data = {
    'Stage': ['Youth Soccer Players', 'High School Players', 'College Players (All Div)', 
              'MLS Roster Spots', 'US-Born MLS', 'USMNT'],
    'Count': [4_000_000, 730_000, 60_000, 850, 400, 70],
    'Percentage': [100, 18.25, 1.5, 0.021, 0.01, 0.0018]
}
funnel_df = pd.DataFrame(funnel_data)

# Visualize with plotly
fig = go.Figure(data=[
    go.Bar(
        y=funnel_df['Stage'],
        x=funnel_df['Count'],
        orientation='h',
        text=[f"{c:,.0f}<br>({p:.4f}%)" for c, p in zip(funnel_df['Count'], funnel_df['Percentage'])],
        textposition='inside',
        marker=dict(color=funnel_df['Percentage'], 
                   colorscale='Reds', 
                   showscale=True,
                   colorbar=dict(title='% of Youth'))
    )
])

fig.update_layout(
    title='US Soccer Career Funnel: Path to Professional Play',
    xaxis_title='Number of Players (log scale)',
    yaxis_title='Career Stage',
    xaxis_type='log',
    height=500,
    showlegend=False
)
fig.show()

print('\nODDS OF REACHING EACH STAGE:')
print(funnel_df.to_string(index=False))

## 2. CAREER COST ANALYSIS: Cost Per Tier (Ages 6-18, 13 Years)

In [ ]:
# Calculate career costs by tier (U6 through U18 = 13 years)
# Tier 1: Recreational (AYSO, USYS rec) - years 1-4, then shift to Tier 2 at U11
# Tier 2: Competitive travel - years 5-8 (U11-U14)
# Tier 3: Elite (ECNL, MLS NEXT) - years 9-13 (U15-U19)

career_cost_scenarios = {
    'Pathway': [
        'Recreational Only (AYSO/USYS Rec)',
        'Rec → Travel (Competitive)',
        'Rec → Travel → Elite (ECNL/MLS NEXT)',
        'Elite Only (U13-U19, MLS NEXT HG)'
    ],
    'Years_in_Rec': [13, 4, 4, 0],
    'Years_in_Travel': [0, 9, 4, 0],
    'Years_in_Elite': [0, 0, 5, 7],
}

# Average costs per tier per year (mid-range from costs.csv)
rec_annual = 550  # AYSO/USYS average
travel_annual = 3_100  # US Club Soccer competitive
elite_annual = 8_000  # ECNL/MLS NEXT mid estimate

cost_scenarios_df = pd.DataFrame(career_cost_scenarios)
cost_scenarios_df['Total_Cost'] = (
    cost_scenarios_df['Years_in_Rec'] * rec_annual +
    cost_scenarios_df['Years_in_Travel'] * travel_annual +
    cost_scenarios_df['Years_in_Elite'] * elite_annual
)

# Visualize cost breakdown
fig = go.Figure()

fig.add_trace(go.Bar(
    x=cost_scenarios_df['Pathway'],
    y=cost_scenarios_df['Years_in_Rec'] * rec_annual,
    name='Recreational',
    marker_color='#2ecc71'
))

fig.add_trace(go.Bar(
    x=cost_scenarios_df['Pathway'],
    y=cost_scenarios_df['Years_in_Travel'] * travel_annual,
    name='Competitive Travel',
    marker_color='#f39c12'
))

fig.add_trace(go.Bar(
    x=cost_scenarios_df['Pathway'],
    y=cost_scenarios_df['Years_in_Elite'] * elite_annual,
    name='Elite (ECNL/MLS NEXT)',
    marker_color='#e74c3c'
))

fig.update_layout(
    barmode='stack',
    title='13-Year Career Cost by Pathway (Ages 6-18)',
    yaxis_title='Total Cost (USD)',
    height=500,
    hovermode='x unified'
)
fig.show()

print('\nCARESR COST BREAKDOWN (13 Years, Ages 6-18):')
print(cost_scenarios_df.to_string(index=False))
print(f'\nCost Per Year Analysis:')
print(f'  Recreational: ${rec_annual:,}/year')
print(f'  Competitive Travel: ${travel_annual:,}/year')
print(f'  Elite (ECNL/MLS NEXT): ${elite_annual:,}/year')

## 3. EXPECTED VALUE CALCULATION: What Are You Actually Paying For?

In [ ]:
# Expected Value Analysis: What's the tangible return?

# Scenario: Family with kid in Elite tier (MLS NEXT HG), spending $40,000 over 6 years (U13-U19)
# What are realistic outcomes?

elite_investment = 48_000  # 6 years × $8,000/year

# Probability-weighted outcomes
outcomes = {
    'Outcome': [
        'No college or pro (most likely)',
        'D3 or NAIA with partial aid',
        'D2 with full athletic aid',
        'D1 with full athletic aid',
        'MLS NEXT Pro → MLS contract',
        'USMNT'
    ],
    'Probability': [0.85, 0.10, 0.03, 0.015, 0.004, 0.0001],
    'Tangible_Benefit': [0, 50_000, 240_000, 280_000, 600_000, 5_000_000],
    'Benefit_Type': [
        'Development/experience only',
        'Partial scholarship (avg $50k/4yr)',
        'Full D2 scholarship (~$60k/year × 4)',
        'Full D1 scholarship (~$70k/year × 4)',
        '6-year MLS career avg earning',
        '10-year USMNT career avg'
    ]
}

outcomes_df = pd.DataFrame(outcomes)
outcomes_df['Expected_Value'] = outcomes_df['Probability'] * outcomes_df['Tangible_Benefit']
outcomes_df['ROI_Multiplier'] = outcomes_df['Tangible_Benefit'] / elite_investment

print('EXPECTED VALUE ANALYSIS: $48,000 Elite Investment (U13-U19)')
print('='*100)
for idx, row in outcomes_df.iterrows():
    print(f"\n{row['Outcome']}")
    print(f"  Probability: {row['Probability']*100:.3f}%")
    print(f"  Tangible Benefit: ${row['Tangible_Benefit']:,}")
    print(f"  Expected Value: ${row['Expected_Value']:,.0f}")
    print(f"  ROI: {row['ROI_Multiplier']:.2f}x")

total_ev = outcomes_df['Expected_Value'].sum()
print(f"\n{'='*100}")
print(f"TOTAL EXPECTED VALUE: ${total_ev:,.0f}")
print(f"Investment: ${elite_investment:,}")
print(f"Expected ROI: {total_ev/elite_investment:.2f}x")
print(f"\nInterpretation: For every $1 spent on elite youth soccer,")
print(f"you expect ~${total_ev/elite_investment:.2f} in tangible returns (scholarships + pro earnings).")

# Visualize outcomes
fig = px.bar(
    outcomes_df,
    x='Outcome',
    y=['Expected_Value'],
    title='Expected Value by Outcome ($48K Elite Investment)',
    labels={'Expected_Value': 'Expected Value (USD)', 'Outcome': 'Outcome'},
    height=500
)
fig.update_layout(showlegend=False)
fig.show()

## 4. SCHOLARSHIP ANALYSIS: The More Realistic ROI

In [ ]:
# NCAA scholarship analysis
# Key stats: 60k college players, 42k scholarship spots (but many unfunded)

# Real scholarship value by division (from research + NCAA data)
scholarship_value = {
    'Division': ['D1 Men', 'D1 Women', 'D2', 'D3', 'NAIA', 'NJCAA'],
    'Programs': [206, 341, 479, 830, ~1800, ~230],
    'Athletes': [6901, 10931, 16872, 25242, ~35000, ~8000],
    'Avg_Scholarship_Value': [35000, 38000, 18000, 0, 15000, 12000],
    'Full_Ride_Percentage': [0.15, 0.20, 0.05, 0, 0.08, 0.03]
}

schol_df = pd.DataFrame(scholarship_value)
schol_df['Total_Aid_Available'] = schol_df['Athletes'] * schol_df['Avg_Scholarship_Value']
schol_df['Probability_of_Getting_Aid'] = (schol_df['Athletes'] * schol_df['Full_Ride_Percentage']) / 60_000

# Calculate expected value of college pathway
college_ev_data = []
for idx, row in schol_df.iterrows():
    four_year_aid = row['Avg_Scholarship_Value'] * 4
    prob = row['Probability_of_Getting_Aid']
    ev = prob * four_year_aid
    college_ev_data.append({
        'Division': row['Division'],
        'Avg_4Yr_Aid': four_year_aid,
        'Probability': prob,
        'Expected_Value': ev
    })

college_ev_df = pd.DataFrame(college_ev_data)
total_college_ev = college_ev_df['Expected_Value'].sum()

print('SCHOLARSHIP PATHWAY ANALYSIS')
print('='*100)
print(schol_df.to_string(index=False))
print(f"\nCollege Expected Value Analysis:")
print(college_ev_df.to_string(index=False))
print(f"\nTotal Expected Value (College Pathway): ${total_college_ev:,.0f}")
print(f"vs Elite Investment: $48,000")
print(f"College ROI: {total_college_ev/48000:.2f}x")

## 5. MLS SALARY REALITY: The Pro Path

In [ ]:
# Real MLS salary distribution from 916 players (MLSPA 2026)
print('MLS SALARY DISTRIBUTION (2026 SEASON, 916 PLAYERS)')
print('='*100)
print(f"Minimum: ${mls_salaries['base_salary'].min():,.0f}")
print(f"25th percentile: ${mls_salaries['base_salary'].quantile(0.25):,.0f}")
print(f"Median: ${mls_salaries['base_salary'].median():,.0f}")
print(f"Mean: ${mls_salaries['base_salary'].mean():,.0f}")
print(f"75th percentile: ${mls_salaries['base_salary'].quantile(0.75):,.0f}")
print(f"90th percentile: ${mls_salaries['base_salary'].quantile(0.90):,.0f}")
print(f"Maximum: ${mls_salaries['base_salary'].max():,.0f}")
print(f"\nTotal League Payroll: ${mls_salaries['base_salary'].sum():,.0f}")

# Career earnings scenarios (Watanabe 2010: avg 2.4yr initial career)
career_scenarios = {
    'Scenario': [
        'Minimum wage (17% of MLS)',
        'Median earner (50th %ile)',
        'Top 25% earner',
        'Top 13% ($1M+)',
        'Superstars (top 3%)'
    ],
    'Annual_Salary': [
        88025,
        325000,
        656750,
        1200000,
        3000000
    ],
    'Career_Years': [2.4, 2.4, 4.0, 6.6, 8.0],
    'Probability_Given_MLS': [0.17, 0.50, 0.25, 0.13, 0.03]
}

career_df = pd.DataFrame(career_scenarios)
career_df['Total_Earnings'] = career_df['Annual_Salary'] * career_df['Career_Years']
career_df['Expected_Value'] = (
    (0.004 / career_df['Probability_Given_MLS'].sum()) * 
    career_df['Probability_Given_MLS'] * 
    career_df['Total_Earnings']
)

print('\n\nPRO CAREER SCENARIOS')
print('='*100)
for idx, row in career_df.iterrows():
    print(f"\n{row['Scenario']}")
    print(f"  Annual: ${row['Annual_Salary']:,}")
    print(f"  Career Length: {row['Career_Years']} years")
print(f"  Total Earnings: ${row['Total_Earnings']:,.0f}")
    print(f"  Probability (given MLS): {row['Probability_Given_MLS']*100:.1f}%")

print(f"\n\nPRO CAREER EXPECTED VALUE (across all scenarios)")
print(f"  Probability of reaching MLS: 0.4% (from 53k MLS NEXT players)")
print(f"  Conditional probability distribution across salary tiers")
print(f"  Total Expected Value: ${career_df['Expected_Value'].sum():,.0f}")

## 6. THE COMPARISON: Where Does Your Money Actually Go?

In [ ]:
# Final ROI Comparison Dashboard
# Family spends $48,000 on elite youth soccer (U13-U19)
# What are the realistic returns?

roi_comparison = {
    'Investment Type': [
        'Elite Youth Soccer (6 years)',
        'Elite + Most Likely Outcome (No Pro/No Scholarship)',
        'Elite + College Scholarship (D1/D2)',
        'Elite + MLS Professional Career',
        'Elite + USMNT Path'
    ],
    'Cost': [48000, 48000, 48000, 48000, 48000],
    'Return_Value': [0, 0, 140000, 300000, 2500000],
    'Probability': [1.0, 0.85, 0.13, 0.004, 0.0001],
    'Expected_Return': [0, 0, 18200, 1200, 250]
}

roi_df = pd.DataFrame(roi_comparison)
roi_df['ROI_Multiplier'] = roi_df['Return_Value'] / roi_df['Cost']
roi_df['Net_Value'] = roi_df['Expected_Return'] - roi_df['Cost']

print('\nFINAL ROI DASHBOARD')
print('='*100)
print('Investment: $48,000 (Elite Youth Soccer, Ages 13-19)')
print(roi_df.to_string(index=False))

print(f"\n\nWHAT THIS MEANS:")
print(f"  Expected return from elite youth soccer: ${roi_df[roi_df['Investment Type'].str.contains('Most Likely')]['Expected_Return'].values[0]:,.0f}")
print(f"  Expected return if scholarship: ${roi_df[roi_df['Investment Type'].str.contains('College')]['Expected_Return'].values[0]:,.0f}")
print(f"  Expected return if pro: ${roi_df[roi_df['Investment Type'].str.contains('Professional')]['Expected_Return'].values[0]:,.0f}")
print(f"\n  The $48k investment expects ${roi_df['Expected_Return'].sum():,.0f} in returns")
print(f"  That's a {roi_df['Expected_Return'].sum()/48000:.2%} expected ROI")

# Visualize the comparison
fig = go.Figure()

fig.add_trace(go.Bar(
    x=roi_df['Investment Type'],
    y=roi_df['Cost'],
    name='Cost',
    marker_color='#e74c3c'
))

fig.add_trace(go.Bar(
    x=roi_df['Investment Type'],
    y=roi_df['Expected_Return'],
    name='Expected Return',
    marker_color='#2ecc71'
))

fig.update_layout(
    barmode='group',
    title='$48,000 Investment: Cost vs Expected Return by Outcome',
    yaxis_title='Value (USD)',
    height=600,
    hovermode='x unified'
)
fig.show()

# The hard truth
print("\n\nTHE HARD TRUTH:")
print(f"  85% chance: $0 return on $48,000 (no college, no pro)")
print(f"  13% chance: $140,000 return (college scholarship)")
print(f"  0.4% chance: $300,000 return (professional career)")
print(f"  0.01% chance: $2.5M return (USMNT)")

## 7. GEOGRAPHIC & ACADEMY STRATIFICATION

In [ ]:
# MLS Academy rankings impact on probability
academy_tiers = {
    'Academy': ['Philadelphia (Elite #1)', 'New England (Elite #2)', 'Chicago (Elite #3)', 
                'Regional Tier (avg 15-20)', 'Weak Regional (KC, Austin)'],
    'Ranking': [1, 2, 3, '15-20', '25-29'],
    'Estimated_MLS_Probability': [0.008, 0.006, 0.005, 0.002, 0.0005],
    'National_Recruiting': ['Yes', 'Yes', 'Limited', 'Local', 'Local'],
    'Resources': ['Top-tier', 'Top-tier', 'Rising', 'Average', 'Limited']
}

academy_df = pd.DataFrame(academy_tiers)

print('\nMLS ACADEMY QUALITY STRATIFICATION')
print('='*100)
print(academy_df.to_string(index=False))
print(f"\nNote: Even 'elite' academies have <1% MLS probability")
print(f"Geography matters: Philadelphia academy 16x better odds than KC academy")
print(f"But absolute odds still require generational talent + luck")

## Key Findings Summary

In [ ]:
print("\n\nKEY FINDINGS & CONCLUSIONS")
print("="*100)
print(f"""
1. THE FUNNEL IS EXTREME
   - 4M youth players → 70 USMNT = 0.0018% probability
   - 4M youth players → 400 US-born MLS = 0.01% probability
   - Even reaching D1 college is only 1.5% probability

2. COST-BENEFIT FOR ELITE TIER ($48,000 investment)
   - Expected value: ${roi_df['Expected_Return'].sum():,.0f}
   - 85% of families get zero tangible return
   - College scholarships materialize for ~13% (if best case)
   - Professional careers for <0.5%

3. COLLEGE IS THE REALISTIC GOAL
   - ~60,000 college players total
   - ~40,000+ scholarship spots (but many unfunded)
   - D1 full rides worth ~$280,000 over 4 years
   - D2 full rides worth ~$240,000 over 4 years
   - Higher probability (13% for elite youth soccer players vs 0.4% MLS)

4. GEOGRAPHY MATTERS SIGNIFICANTLY
   - Philadelphia academy: 8x better MLS odds than average
   - But still <1% even in elite academies
   - Coastal advantage: better recruiting, resources, competition

5. PAY-TO-PLAY CREATES EQUITY GAPS
   - Elite tier: $8,000/year × 6 years = $48,000 total
   - Locks out 92% of US youth (median family income $60k)
   - Lower-income talent never reaches coach evaluation
   - AI scouting (video-based) could disrupt this gatekeeping

6. THE INTERNATIONAL ADVANTAGE
   - 53% of MLS is international players
   - Foreign-born more likely to reach and stay in MLS
   - US-developed talent often leaves for Europe (better pay)
   - Brain drain: Pulisic (AC Milan), McKennie (Juventus), etc.

7. WHAT ACTUALLY WORKS
   - Recreational soccer is healthiest (low cost, high participation, joy)
   - Competitive travel tier justifies cost only if college goal is realistic
   - Elite tier ROI negative unless kid identified as elite by age 13-14
   - Best value: competitive club + school soccer (dual pathways)
""")